# Dimensionality Reduction and Clustering
## What changes when we change the feature space?

**Laboratory 3 — individual assignment — maximum 10 points**

In Practical 08 you used PCA to represent observations in fewer dimensions. In Practical 09 you
used K-means and DBSCAN to group observations without class labels.

Now you will put these ideas together on **a different real dataset**. Your task is to find out
whether the grouping changes when the same observations are represented in the original feature
space, in a PCA space, or through a density-based method.

### Dataset

We use the **Breast Cancer Wisconsin (Diagnostic)** dataset from the UCI Machine Learning Repository,
distributed through `scikit-learn`. It contains **569 observations and 30 numeric features** derived
from digitized images of fine-needle aspirates of breast masses. The features describe measured
properties of cell nuclei.

The source has reference diagnosis labels, but they are stored in a **separate file**. You will use
these labels only **after** completing the unsupervised analysis. A diagnosis label is not an input
feature for clustering or PCA.

> This is a historical research dataset for learning numerical methods, not a clinical validation
> dataset. The exercise does not produce a diagnostic tool or support medical decisions.

Read [`SOURCES_AND_DATA_DICTIONARY.md`](SOURCES_AND_DATA_DICTIONARY.md) for the source,
license, feature groups, and known limitations.

### Submission

Submit one notebook named `Lab 03 - <Surname> <Name>.ipynb`.

Your notebook must contain the analysis, figures, and short written answers. It must run with
**Restart Kernel → Run All**, using only relative paths. Keep the source CSV files unchanged.

### What you will investigate

1. How much does feature scaling affect the geometry of the data?
2. How much variation does PCA retain in two dimensions and in a higher-dimensional projection?
3. Does K-means produce similar groups before and after PCA?
4. Does DBSCAN see comparable groups, or mostly one dense region and noise?
5. How do the resulting partitions relate to the **held-back** reference labels?

A strong solution includes numerical evidence and limitations, not just attractive plots.

---
## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

DATA = Path("data")
MEASUREMENTS = DATA / "measurements.csv"
REFERENCE = DATA / "reference_labels.csv"

for file in (MEASUREMENTS, REFERENCE):
    if not file.is_file():
        raise FileNotFoundError(f"Missing {file}. Run the notebook from the extracted package folder.")

measurements = pd.read_csv(MEASUREMENTS)
assert measurements["sample_id"].is_unique

sample_ids = measurements["sample_id"].copy()
X = measurements.drop(columns="sample_id")

print("Observations:", len(X))
print("Numeric features:", X.shape[1])
print("Source file:", MEASUREMENTS)
X.head()

---
## 1. Inspect the feature space

Before fitting any algorithm, inspect the measurements. Report missing values, data types, feature
ranges and at least two pairs of noticeably correlated features. Visualize something relevant to the
scaling decision (for example, a spread/range comparison or selected feature distributions).

Explain **why raw Euclidean distance is difficult to interpret** when different features have very
different numerical scales. Some features describe related geometric properties, so correlation and
redundancy also deserve attention.

Create a standardized feature matrix using `StandardScaler`. Keep the original DataFrame unchanged.
Use the standardized matrix for the main PCA and clustering experiments.

This lab is descriptive, not a model evaluated on a held-out prediction set. You may fit the
scaler to the provided observations. In predictive work, scaling must be fitted using training data
only.

In [ ]:
# Your solution starts here.  # Audit, visualization, and standardization

**Your findings:** *Summarize the scaling choice and one notable feature relationship here.*

---
## 2. PCA: 2D is not the same as keeping most of the variation

Fit PCA on the standardized features.

- Plot the cumulative explained-variance ratio.
- Find the **smallest** number of components needed for at least **90%** cumulative explained
  variance. Report that number and the percentage retained by the first two components.
- Produce a two-dimensional PCA scatter plot, **without coloring points by diagnosis**.
- Inspect the strongest contributions to PC1 or PC2. Explain what the signs and magnitudes of
  the loadings suggest, without claiming that one component is a clinical factor.

Keep both representations for the next section: a 2D projection for visualization and the
projection retaining at least 90% variance for clustering.

**Important:** the 90% threshold measures retained *variance*, not the preservation of every
useful pattern. PCA does not know the reference labels.

In [ ]:
# Your solution starts here.  # PCA, variance, projections, and loadings

**Your findings:** *How many components were needed? What is lost when you show only two?*

---
## 3. K-means: do the groups survive a change of representation?

Cluster the **standardized original features** with K-means for `k = 2, 3, 4`.

For each `k`, report the cluster sizes and silhouette score. Use a fixed `random_state` and
explicit `n_init` to make the experiment repeatable. Choose a sensible `k` based on this
**unsupervised evidence** and explain the choice; do not use reference labels yet.

Then apply K-means with the **same chosen `k`** to the PCA representation retaining at least 90%
variance. Compare the two partitions numerically (for example, with Adjusted Rand Index applied
to the **two sets of cluster assignments**). Plot the full-feature and PCA-based assignments in the
same 2D PCA coordinate system so the comparison is visually consistent.

Silhouette scores from different feature spaces are not directly interchangeable as a universal
quality ranking: the distances themselves changed. Inertia also should not be compared across
representations as if it had the same scale.

In [ ]:
# Your solution starts here.  # K-means trials, 90%-PCA clustering, and comparison

**Your findings:** *Which partition did you choose? How much did assignments change after PCA?*

---
## 4. DBSCAN: what counts as a dense neighbourhood?

Run DBSCAN on the **2D PCA coordinates**. Try at least three values of `eps` with a fixed
`min_samples` value and report:

- the number of clusters (**excluding** label `-1`);
- the fraction of observations marked as noise (`-1`);
- the sizes of the clusters that remain.

The 2D projection is used here deliberately so density neighbourhoods can be inspected visually.
Make one plot for a configuration you consider informative and explain what happens when `eps`
changes.

**A valid result may be one large cluster, several small clusters, or many noise points.** Do not
force DBSCAN to return exactly two groups just because the reference file has two diagnosis labels.

If fewer than two non-noise clusters exist, a conventional silhouette score for those assigned
points is undefined. Do not invent a score or treat noise as an ordinary discovered cluster without
explaining that decision.

In [ ]:
# Your solution starts here.  # DBSCAN parameter comparison and plot

**Your findings:** *What neighbourhood assumption seems to matter most?*

---
## 5. Only now: compare with the reference labels

The second file contains the original dataset's diagnostic categories. They are **external
information** about the samples, not a definition of what an unsupervised cluster must be.

Use `sample_id` to align them with the observations. Do not assume that two CSV files will always
have the same row order.

In [ ]:
reference = pd.read_csv(REFERENCE)
assert reference["sample_id"].is_unique

aligned_reference = (
    measurements[["sample_id"]]
    .merge(reference, on="sample_id", how="left", validate="one_to_one")
)

assert aligned_reference["diagnosis"].notna().all()
print(aligned_reference["diagnosis"].value_counts())
aligned_reference.head()

Compare your selected K-means partition(s) with the reference labels using:

1. a contingency table (`pd.crosstab`);
2. Adjusted Rand Index (ARI), which does not depend on the arbitrary numerical names of clusters.

You may also compare a DBSCAN result, but describe how noise points (`-1`) were handled. If you
evaluate only non-noise points, report how many observations you excluded.

Explain one agreement and one disagreement between clustering and reference classes.

**Do not tune your unsupervised model after seeing these labels and then describe the ARI as an
independent, untouched validation result.** It becomes label-informed model selection at that point.

This exercise does **not** estimate diagnostic performance. Differences between cluster membership
and diagnosis are not automatically model errors, nor do agreements establish clinical utility.

In [ ]:
# Your solution starts here.  # Contingency table and external evaluation

**Your findings:** *What did the labels reveal that internal metrics alone could not?*

---
## 6. Final comparison and submission

Summarize your experiments in **one compact table**. Include the representation, method,
parameters, number of clusters, and at least one appropriate diagnostic. Report noise separately
for DBSCAN. State clearly which numbers are comparable and which are not.

Export a CSV called `results/cluster_assignments.csv` containing `sample_id` and the assignments
from your selected K-means models and one DBSCAN configuration. Add PCA coordinates if useful;
**do not overwrite the original source files**.

Finish with **150–220 words** covering:

- what scaling changed and how much PCA compressed the dataset;
- whether the K-means groups were stable across representations;
- what DBSCAN found, including noise and parameter sensitivity;
- what the reference labels did and did not establish;
- which representation and method you would use for an **exploratory grouping task**, and why.

The goal is not to declare a universal winner. It is to defend a result with an understanding of the
representation and comparison rule.

In [ ]:
# Your solution starts here.  # Summary table and CSV export

**Final report:** *Write 150–220 words here, referring to your own tables and figures.*

---
## Grading — 10 points

| Criterion | Points | What earns full credit |
|---|---:|---|
| Inspection and preprocessing | **1.5** | Correct numeric audit, clear scaling justification, no label leakage |
| PCA | **2.0** | Explained variance, justified component count, plot and loadings interpretation |
| K-means experiments | **2.0** | Repeatable comparisons, cluster sizes, sensible `k`, full-space vs PCA analysis |
| DBSCAN experiments | **1.5** | Meaningful parameter sweep, honest reporting of noise and cluster count |
| External evaluation | **1.5** | Proper label alignment, contingency table / ARI and cautious interpretation |
| Results and reproducibility | **1.5** | Clear figures, final comparison, exported assignments and fresh-kernel execution |
| **Total** | **10.0** | |

### Before submitting

- [ ] The reference labels were not used for scaling, PCA, `k` selection, or clustering.
- [ ] The PCA plot and variance threshold are consistent with the fitted model.
- [ ] All K-means and DBSCAN results include the relevant parameters and sample counts.
- [ ] `-1` is treated as noise when interpreting DBSCAN.
- [ ] At least one clustering result is compared with reference labels **only after** the unsupervised work.
- [ ] `results/cluster_assignments.csv` is produced by notebook code.
- [ ] Original CSV files are unchanged; **Restart Kernel → Run All** succeeds.

### Source and documentation

- [UCI — Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)
- [scikit-learn — `load_breast_cancer`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)
- [scikit-learn — PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [scikit-learn — Clustering](https://scikit-learn.org/stable/modules/clustering.html)